In [1]:
import pandas as pd
import os
import math

# ==============================================================================
# 1. CẤU HÌNH PHÂN LOẠI FILE
# ==============================================================================
# Danh sách các file THƠ (Cần cắt theo khổ 4 dòng)
POETRY_FILES = [
    "Kieu_1866.xlsx",
    "LucVanTien.xlsx",
    "chinh_phu_ngam_khuc.xlsx",
    "ho_xuan_huong.xlsx",
    "quoc_am_thi_tap.xlsx"
]

# Danh sách các file VĂN XUÔI
PROSE_FILES = [
    "DVSKTT-1_Quyen_Thu.xlsx", 
    "DVSKTT-2_Ngoai_ky_toan_thu.xlsx",
    "DVSKTT-3_Ban_ky_toan_thu.xlsx",
    "DVSKTT-4_Ban_ky_thuc_luc.xlsx",
    "DVSKTT-5_Ban_ky_tuc_bien.xlsx",
    "Tam_quoc_3.xlsx"
]

INPUT_DIR = "/kaggle/input/rawdata/"
OUTPUT_DIR = "/kaggle/working/"
TEST_RATIO = 0.1
RANDOM_STATE = 42

# ==============================================================================
# 2. HÀM CẮT CHUYÊN BIỆT
# ==============================================================================

INPUT_DIR = "/kaggle/input/rawdata/"
OUTPUT_DIR = "/kaggle/working/"
TEST_RATIO = 0.1
RANDOM_STATE = 42

# ==============================================================================
# 2. CÁC HÀM CẮT DATA
# ==============================================================================
def split_poetry_block(df, ratio=0.1, group_size=4):
    """Cắt Thơ: Giữ nguyên khổ 4 dòng"""
    total = len(df)
    clean_len = total - (total % group_size) # Làm tròn xuống cho chia hết cho 4
    df = df.iloc[:clean_len]
    
    n_test = math.ceil((clean_len * ratio) / group_size) * group_size
    split_idx = clean_len - n_test
    
    return df.iloc[:split_idx], df.iloc[split_idx:]

def split_prose_tail(df, ratio=0.1):
    """Cắt Văn: Cắt 10% đuôi"""
    
    split_idx = int(len(df) * (1 - ratio))
    return df.iloc[:split_idx], df.iloc[split_idx:]

# ==============================================================================
# 3. THỰC THI
# ==============================================================================
train_buckets = []
test_poetry_buckets = []
test_prose_buckets = []

print("🚀 Bắt đầu phân loại và chia dữ liệu...")

all_files = [f for f in os.listdir(INPUT_DIR) if f.endswith('.xlsx')]

for file_name in all_files:
    file_path = os.path.join(INPUT_DIR, file_name)
    try:
        df = pd.read_excel(file_path)
        df = df.dropna(subset=['Text'])
        
        # Thêm cột nguồn để dễ trace
        df['Source'] = file_name
        
        train_df = None
        test_df = None
        type_lbl = ""

        # --- LOGIC PHÂN LOẠI ---
        if file_name in POETRY_FILES:
            type_lbl = "THƠ"
            train_df, test_df = split_poetry_block(df, TEST_RATIO, group_size=4)
            if test_df is not None: test_poetry_buckets.append(test_df)
            
        elif file_name in PROSE_FILES:
            type_lbl = "VĂN"
            train_df, test_df = split_prose_tail(df, TEST_RATIO)
            if test_df is not None: test_prose_buckets.append(test_df)
            
        else:
            print(f"Bỏ qua file lạ: {file_name}")
            continue

        if train_df is not None:
            train_buckets.append(train_df)
            print(f" {file_name} [{type_lbl}]: Train {len(train_df)} | Test {len(test_df)}")

    except Exception as e:
        print(f" Lỗi {file_name}: {e}")

# ==============================================================================
# 4. GỘP VÀ LƯU FILE (3 FILE RIÊNG BIỆT)
# ==============================================================================
if train_buckets:
    # 1. TRAIN: Gộp tất cả + SHUFFLE 
    final_train = pd.concat(train_buckets, ignore_index=True)
    final_train = final_train.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
    
    # 2. TEST THƠ: Gộp + GIỮ NGUYÊN THỨ TỰ (để test ghép cặp)
    final_test_poetry = pd.concat(test_poetry_buckets, ignore_index=True) if test_poetry_buckets else pd.DataFrame()
    
    # 3. TEST VĂN: Gộp + GIỮ NGUYÊN THỨ TỰ (để test ngữ cảnh)
    final_test_prose = pd.concat(test_prose_buckets, ignore_index=True) if test_prose_buckets else pd.DataFrame()

    print("-" * 50)
    print(f" TỔNG KẾT:")
    print(f"   1. Final Train (Mix):   {len(final_train)} dòng")
    print(f"   2. Test Thơ (Poetry):   {len(final_test_poetry)} dòng")
    print(f"   3. Test Văn (Prose):    {len(final_test_prose)} dòng")
    
    # Lưu file
    final_train.to_excel(os.path.join(OUTPUT_DIR, "final_train.xlsx"), index=False)
    final_test_poetry.to_excel(os.path.join(OUTPUT_DIR, "test_poetry.xlsx"), index=False)
    final_test_prose.to_excel(os.path.join(OUTPUT_DIR, "test_prose.xlsx"), index=False)
    
    print("\n Đã lưu 3 file riêng biệt vào folder output.")
else:
    print("❌ Không có dữ liệu.")

🚀 Bắt đầu phân loại và chia dữ liệu...
   ✅ ho_xuan_huong.xlsx [THƠ]: Train 300 | Test 36
   ✅ Tam_quoc_3.xlsx [VĂN]: Train 12248 | Test 1361
   ✅ Kieu_1866(1).xlsx [THƠ]: Train 2148 | Test 240
   ✅ DVSKTT-2_Ngoai_ky_toan_thu.xlsx [VĂN]: Train 1532 | Test 171
   ✅ quoc_am_thi_tap (1).xlsx [THƠ]: Train 1656 | Test 188
   ✅ DVSKTT-1_Quyen_Thu.xlsx [VĂN]: Train 500 | Test 56
   ✅ chinh_phu_ngam_khuc.xlsx [THƠ]: Train 364 | Test 44
   ✅ DVSKTT-4_Ban_ky_thuc_luc.xlsx [VĂN]: Train 5488 | Test 610
   ✅ LucVanTien.xlsx [THƠ]: Train 1852 | Test 208
   ✅ DVSKTT-3_Ban_ky_toan_thu.xlsx [VĂN]: Train 7955 | Test 884
   ✅ DVSKTT-5_Ban_ky_tuc_bien.xlsx [VĂN]: Train 3226 | Test 359
--------------------------------------------------
📊 TỔNG KẾT:
   1. Final Train (Mix):   37269 dòng
   2. Test Thơ (Poetry):   716 dòng
   3. Test Văn (Prose):    3441 dòng

💾 Đã lưu 3 file riêng biệt vào folder output.
